# 01 - Hola Mundo con Gemini + LangChain

Primer contacto con la API de Google Gemini usando LangChain.

**Objetivos:**
- Cargar la API key desde `.env`
- Instanciar el modelo `gemini-2.5-flash`
- Enviar un mensaje simple y recibir una respuesta

## 1. Cargar variables de entorno

In [ ]:
import os
from dotenv import load_dotenv

# Sube un nivel desde notebooks/ para encontrar el .env en la raíz
load_dotenv(dotenv_path="../.env")

api_key = os.getenv("GOOGLE_API_KEY")
print("API key cargada:", "✓" if api_key else "✗ No encontrada")

## 2. Instanciar el modelo Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.7,
)

print("Modelo listo:", llm.model)

## 3. Hola Mundo — primera llamada al modelo

In [ ]:
from langchain_core.messages import HumanMessage

mensaje = HumanMessage(content="Hola! Preséntate brevemente en español.")

respuesta = llm.invoke([mensaje])

print("Respuesta:", respuesta.content)
print("\n--- Metadata ---")
print("Modelo usado:", respuesta.response_metadata.get("model_version", "N/A"))
print("Tokens usados:", respuesta.usage_metadata)

## 4. Usando un prompt con string directo (forma corta)

In [ ]:
# LangChain también acepta un string directo
respuesta2 = llm.invoke("¿Cuál es la capital de México? Responde en una sola oración.")
print(respuesta2.content)

## 5. Prompt Templates

Las plantillas permiten reutilizar prompts con variables dinámicas.

| Clase | Cuándo usarla |
|---|---|
| `PromptTemplate` | Prompts de texto plano con variables |
| `ChatPromptTemplate` | Conversaciones con roles (system / human / ai) |

### 5a. PromptTemplate — texto plano

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    "Explica {tema} en máximo {oraciones} oraciones, en español."
)

# Renderiza el prompt con valores concretos
prompt_listo = template.format(tema="la fotosíntesis", oraciones=2)
print(prompt_listo)

respuesta3 = llm.invoke(prompt_listo)
print("\nRespuesta:", respuesta3.content)

### 5b. ChatPromptTemplate — con roles (few-shot)

Incluye un ejemplo de entrada/salida antes de la pregunta real para guiar al modelo.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a geography expert that returns the colors present in a country's flag."),
        ("human", "France"),
        ("ai", "blue, white, red"),
        ("human", "{country}"),
    ]
)

# Genera los mensajes listos para enviar al modelo
mensajes = prompt_template.invoke({"country": "Mexico"})
print(mensajes.messages)

respuesta4 = llm.invoke(mensajes)
print("\nColores de la bandera de México:", respuesta4.content)

### 5c. FewShotChatMessagePromptTemplate — múltiples ejemplos

Cuando tienes varios ejemplos, es más limpio separarlos en una lista y usar `FewShotChatMessagePromptTemplate`. Esto te permite agregar o quitar ejemplos sin tocar la estructura del prompt.

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# Lista de ejemplos — fácil de ampliar
ejemplos = [
    {"country": "France",  "colors": "blue, white, red"},
    {"country": "Italy",   "colors": "green, white, red"},
    {"country": "Japan",   "colors": "white, red"},
    {"country": "Germany", "colors": "black, red, yellow"},
]

# Plantilla que define cómo se ve cada ejemplo
ejemplo_template = ChatPromptTemplate.from_messages([
    ("human", "{country}"),
    ("ai",    "{colors}"),
])

# Ensambla todos los ejemplos automáticamente
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=ejemplo_template,
    examples=ejemplos,
)

# Prompt final: system + ejemplos + pregunta real
prompt_final = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert that returns the colors present in a country's flag."),
    few_shot_prompt,
    ("human", "{country}"),
])

# Prueba con un país
mensajes = prompt_final.invoke({"country": "Brazil"})

# Muestra cómo quedó el prompt completo
for m in mensajes.messages:
    print(f"[{m.type}] {m.content}")

print("\n--- Respuesta del modelo ---")
respuesta5 = llm.invoke(mensajes)
print(respuesta5.content)